Direct YML Files analysis or keywords detection

in this version we review the yml files for general instr test signal and group the detections based on a confidence level label
this detection will be used in our overall repo V2.0

improving the code based on the findings of 

it improve V2.0 to do not detect instru signal only when "test_trigger: gradle via variable hint" as V3.0

segregating flutter integration test which its execution environment is not android style - V4.0

also Updated regex to improve (reduce FP) in detection in cat usage instead of ConnectedAndroidTest




In [1]:
# -*- coding: utf-8 -*-
"""
Scan CI YAML files for instrumentation-testing signals and output:
filename, full_name, ci_platform, instru_t_ci_signal, confidence, confidence_reason,
execution_environment, test_invocation, flutter_integ_t_signal, flutter_integ_t_d

Highlights
- 'adb wait-for-device' → Emulator
- Strong real device ONLY: 'adb -s <serial> (physical)'
- Uses both 'prefix' (./gradlew ...) and 'anywhere' scans
- Confidence_reason lists BOTH test_trigger and device_setup when present
- Derives execution_environment & test_invocation
- Extracts Flutter device via -d <device>, or infers iOS/Android/Desktop when possible
"""

import re
import pandas as pd
from pathlib import Path
from typing import List, Pattern, Tuple, Dict, Any, Iterable

# === CONFIG ===
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")
TESTS_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Test_Files")
OUTPUT_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_YML_Files_V4.0.csv")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
TESTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# === Helpers ===
def extract_full_name_from_file(filename: str) -> str:
    fname = filename.lower()
    if "__" in fname:
        return fname.split("__", 1)[0]
    return Path(fname).stem

def extract_ci_platform(filename: str) -> str:
    """owner.repo__github++file.yml  -> github"""
    fname = filename.lower()
    if "__" in fname and "++" in fname:
        return fname.split("__", 1)[1].split("++", 1)[0]
    return ""

def build_test_presence_index(tests_dir: Path) -> set[str]:
    present = set()
    for f in tests_dir.glob("*"):
        if not f.is_file():
            continue
        name = f.name.lower()
        if "__" in name:
            repo_key = name.split("__", 1)[0]
            present.add(repo_key)
    return present

ANDROIDTEST_PRESENT = build_test_presence_index(TESTS_DIR)

def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: Iterable[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

def normalize_block_keys(text: str) -> str:
    # Single-line: "run: ./gradlew ..." -> "./gradlew ..."
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\|)\s*(.+)$', r'\2', text)
    # Multiline: drop only the key line (keep following block)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*\|?\s*$', '', text)
    return text

# Gradle command prefix normalizer (env/sudo/bash/cd && ./gradlew)
GRADLE_PREFIX = (
    r'^\s*'
    r'(?:\S+=\S+\s+)*'
    r'(?:sudo\s+)?'
    r'(?:(?:bash|sh)\s+-c[l]?\s+[\'"]?)?'
    r'(?:[^#\n;]*?&&\s+)?'
    r'(?:cd\s+\S+\s+&&\s+)?'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?'
)
GRADLE_ANYWHERE = r'(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*'
NON_TEST_PREFIX = r'(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)'

# === Device/Trigger sources ===
DEVICE_SOURCES = [
    # Strong REAL device (only)
    ("Real_Device", "adb -s <serial> (physical)", [r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b']),
    # Emulator/AVD
    ("Emulator", "adb -s emulator-serial", [
        r'(?m)^\s*adb\s+-s\s+emulator-\d+\b',
        r'(?m)^\s*adb\s+-s\s+(?:localhost|127\.0\.0\.1):\d+\b',
    ]),
    ("Emulator", "adb wait-for-device", [r'(?mi)^\s*adb\s+wait[- ]?for[- ]?device\b']),
    # FIX 1: allow space after -avd, and support "@name"
    ("Emulator", "emulator -avd/@", [
        r'^[^\n]*\bemulator\b[^\n]*(?:-avd\s+\S+|@\S+)'
    ]),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^\s*(?:\./)?android-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh", [r'(?m)^\s*start-emulator\.sh\b']),
    # FIX 2: match anywhere on the line (handle pipes like "echo no | android create avd")
    ("Emulator", "android create avd", [
        r'\bandroid\b[^\n]*\bcreate\s+avd\b'
    ]),
    ("Emulator", "circle-android wait-for-boot",[r'(?m)^\s*circle-android\s+wait-for-boot\b']),
    ("Emulator", "reactivecircus runner", [r'uses:\s*reactivecircus/android-emulator-runner']),
    ("Emulator", "sys-img component", [
        r'(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
        r'(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "avdmanager", [r'(?m)^\s*\S*avdmanager\b']),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r'^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
    ]),
    ("Emulator", "api-level", [r'\bapi[-_ ]?level\b\s*:?\s*\d{2}']),
    ("Emulator", "abi/arch",  [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image", [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name", [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),

    # GMD
    ("GMD", "managedDevices DSL",       [r'\bmanageddevices?\b']),
    ("GMD", "ManagedVirtualDevice DSL", [r'\bmanagedvirtualdevice\b|\bcom\.android\.build\.api\.dsl\.ManagedVirtualDevice\b']),
    ("GMD", "GMD task mentions",        [r'\bmanageddevice\w*androidtest\b']),
    ("GMD", "GHA gradle args for GMD",  [
        r'(?m)^\s*arguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
        r'(?m)^\s*tasks?\s*:\s*[:\w-]*manageddevice\w*androidtest\b'
    ]),

    # Third-party labs
    ("Third_Party_Lab", "gcloud firebase", [r'(?m)^\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",        [r'(?m)^\s*saucectl(\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack", [r'\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",  [r'(?m)^\s*appcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",   [r'(?m)^\s*maestro\s+cloud\b']),
]
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]

TRIGGER_SOURCES_PRIMARY = [
    ("Gradle",  "connectedAndroidTest",                 [rf'(?m){GRADLE_PREFIX}[^\n]*\bconnectedandroidtest\b']),
    ("Gradle",  "connected.*Android.*",                 [rf'(?m){GRADLE_PREFIX}[^\n]*\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b']),
    ("Gradle",  "connectedCheck",                       [rf'(?m){GRADLE_PREFIX}\s+(?::[\w-]+:)*connectedcheck\b']),
    ("Gradle",  "connectedAndroidTest (abbr)",          [rf'(?mi){GRADLE_PREFIX}[^\n]*?(?<!\$\()(?<!`)(?<!\S)(?::[\w-]+:)*cat(?!\S)']),
    ("Gradle",  "deviceCheck",                          [rf'(?mi){GRADLE_PREFIX}\s+(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle",  "managedDevice AndroidTest",            [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),
    ("Gradle",  "variant/device AndroidTest",           [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b']),
    ("Gradle",  "plain androidTest",                    [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?::[\w-]+:)*androidtest\b']),
    ("Gradle",  "Spoon",                                [rf'(?mi){GRADLE_PREFIX}[^\n]*\bspoon(?:\w*androidtest)?\b']),
    ("Gradle",  "Marathon",                             [rf'(?mi){GRADLE_PREFIX}[^\n]*\bmarathon(?:\w*androidtest)?\b']),
    ("ADB",     "am instrument",                        [r'(?mi)^[^\n]*\bam\s+instrument\b']),
    ("Third_Party_Lab", "gcloud firebase",              [r'(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "flank",                        [r'(?mi)^[^\n]*\bflank\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",                     [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run",                [r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b']),
    # Flutter triggers
    ("Flutter", "flutter drive",                        [r'(?mi)^[^\n]*\bflutter\s+drive\b']),
    ("Flutter", "flutter test (integration_test)",      [r'(?mi)^[^\n]*\bflutter\s+test\b[^\n]*\bintegration_test\b']),
    ("Flutter", "dart test (integration_test)",         [r'(?mi)^[^\n]*\bdart\s+test\b[^\n]*\bintegration_test\b']),
]
TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)",                  [rf'(?mi){GRADLE_ANYWHERE}\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b']),
    ("Gradle", "connectedAndroidTest (anywhere)",       [rf'(?mi){GRADLE_ANYWHERE}\bconnectedandroidtest\b']),
    ("Gradle", "deviceCheck (anywhere)",                [rf'(?mi){GRADLE_ANYWHERE}\b(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest (anywhere)",  [rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),
    ("Gradle", "variant/device AndroidTest (anywhere)", [rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b']),
    ("Gradle", "Spoon (anywhere)",                      [rf'(?mi){GRADLE_ANYWHERE}\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon (anywhere)",                   [rf'(?mi){GRADLE_ANYWHERE}\bmarathon(?:\w*androidtest)?\b']),
]
TRIGGER_PATTERNS_PRIMARY  = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_PRIMARY]
TRIGGER_PATTERNS_ANYWHERE = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_ANYWHERE]

# GHA gradle inputs; avoid $(cat ...) or backticks
GHA_GRADLE_INPUTS = compile_any([
    r'(?mi)^\s*arguments\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*arguments\s*:\s*[\w:-]*androidtest\b',
    r'(?mi)^\s*tasks?\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*tasks?\s*:\s*[\w:-]*androidtest\b',
    r'(?mi)^\s*arguments\s*:\s*[^$`\n]*?(?::[\w-]+:)*cat(?:\s|$)',
    r'(?mi)^\s*tasks?\s*:\s*[^$`\n]*?(?::[\w-]+:)*cat(?:\s|$)',
])

# Emulator vs Real reconciliation helpers
EMULATOR_STRONG_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner", "avdmanager",
    "sdkmanager system-images/emulator", "adb -s emulator-serial", "adb wait-for-device",
    # FIX 3: treat as strong
    "android create avd",
}
REAL_DEVICE_STRONG_LABELS = {"adb -s <serial> (physical)"}
REAL_DEVICE_GENERIC_ADB = set()

STRONG_DEVICE_LABELS = EMULATOR_STRONG_LABELS | REAL_DEVICE_STRONG_LABELS
WEAK_DEVICE_LABELS = {"api-level", "abi/arch", "target image", "device name", "sys-img component"}

# --- Flutter device extraction helpers ---
FLUTTER_CMD_LINE_RE = re.compile(r'(?mi)^\s*flutter\s+(?:drive|test)\b[^\n]*')
FLUTTER_DEVICE_FLAG_RE = re.compile(r'(?i)\s+-d\s+(?P<dev>"[^"]+"|\'[^\']+\'|\S+)')
LINUX_HEADLESS_HINTS_RE = re.compile(r'(?mi)^\s*(xvfb-run|export\s+DISPLAY=|sudo\s+Xvfb)\b')

IOS_SIM_HINTS = compile_any([
    r'uses:\s*futureware-tech/simulator-action@',
    r'\bxcrun\s+simctl\b',
    r'\biphonesimulator\b',
    r'\bdestination\b[^\n]*platform=iOS',
    r'(?mi)^\s*model\s*:\s*["\']?\s*(iphone|ipad)\b',
])

# ---------- job splitter (heuristic)
JOBS_ANCHOR_RE = re.compile(r'(?m)^(?P<indent>\s*)jobs\s*:\s*$')
ANY_KEY_RE     = re.compile(r'(?m)^(?P<indent>\s*)(?P<name>[\w-]+)\s*:\s*$')

def split_jobs_blocks(raw: str) -> List[Tuple[str, str]]:
    m = JOBS_ANCHOR_RE.search(raw)
    if not m:
        return [("__whole__", raw)]
    jobs_indent = len(m.group("indent"))
    lines = raw.splitlines(True)
    start_idx = raw[:m.end()].count("\n")
    candidates = []
    for i in range(start_idx, len(lines)):
        lm = ANY_KEY_RE.match(lines[i])
        if not lm:
            continue
        indent = len(lm.group("indent"))
        if indent > jobs_indent:
            candidates.append((i, indent, lm.group("name")))
    if not candidates:
        return [("__whole__", raw)]
    min_indent = min(indent for _, indent, _ in candidates)
    job_headers = [(i, name) for (i, indent, name) in candidates if indent == min_indent]
    if not job_headers:
        return [("__whole__", raw)]
    blocks = []
    header_indices = [i for i, _ in job_headers] + [len(lines)]
    for idx in range(len(job_headers)):
        i, name = job_headers[idx]
        j = header_indices[idx + 1]
        block_text = "".join(lines[i:j])
        blocks.append((name, block_text))
    return blocks

def collect_hits_with_groups(patterns: List[Tuple[str, str, List[Pattern]]], text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl); groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

def filter_weak_device_hints(labels, groups):
    if not (set(labels) & STRONG_DEVICE_LABELS):
        labels = [l for l in labels if l not in WEAK_DEVICE_LABELS]
        if not labels:
            groups = []
    return labels, groups

def reconcile_emulator_vs_real(labels, groups):
    lbls = set(labels)
    if lbls & EMULATOR_STRONG_LABELS:
        # Drop generic real-device ADB (none at present, kept for future use)
        lbls -= REAL_DEVICE_GENERIC_ADB
        labels = [l for l in labels if l in lbls]
        if "Real_Device" in groups:
            has_real_after = bool(set(labels) & REAL_DEVICE_STRONG_LABELS)
            if not has_real_after:
                groups = [g for g in groups if g != "Real_Device"]
    return labels, groups

# === Main scan ===
rows: List[Dict[str, Any]] = []

for f in sorted(CONFIG_DIR.iterdir()):
    if not f.is_file():
        continue
    if f.suffix.lower() not in (".yml", ".yaml"):
        continue

    filename = f.name
    full_name = extract_full_name_from_file(filename)
    ci_platform = extract_ci_platform(filename)

    try:
        raw = f.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        raw = ""

    # per-file aggregates
    file_trigger_labels: List[str] = []
    file_device_labels:  List[str] = []
    file_trigger_groups: List[str] = []
    file_device_groups:  List[str] = []
    file_has_test_trigger = False
    file_has_device_with_group = False
    file_gradle_present_any = False
    file_instru_signal_any = False
    file_flutter_devices: List[str] = []

    for job_name, job_raw in split_jobs_blocks(raw):
        content = strip_comments(job_raw)
        content = re.sub(r'(?m)^\s*-\s*', '', content)
        content = normalize_block_keys(content)

        # Device & trigger detection (prefix-first)
        dev_labels, dev_groups = collect_hits_with_groups(DEVICE_PATTERNS, content.lower())
        dev_labels, dev_groups = filter_weak_device_hints(dev_labels, dev_groups)
        dev_labels, dev_groups = reconcile_emulator_vs_real(dev_labels, dev_groups)

        trig_labels, trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, content.lower())

        # gradle-build-action inputs
        if any_match(GHA_GRADLE_INPUTS, content):
            trig_labels = unique_preserve(trig_labels + ["gha gradle arguments"])
            trig_groups = unique_preserve(trig_groups + ["Gradle"])

        # Fallback "anywhere" scan if needed
        if not trig_labels:
            fallback = strip_comments(job_raw)
            fallback = re.sub(r'(?m)^\s*-\s*', '', fallback)
            fallback = re.sub(r'(?mi)^\s*(?:command|run|script)\s*:\s*\|?\s*', '', fallback)
            fallback = re.sub(r'(?m)^\s*sudo\s+', '', fallback)

            if not dev_labels:
                fb_dev_labels, fb_dev_groups = collect_hits_with_groups(DEVICE_PATTERNS, fallback.lower())
                fb_dev_labels, fb_dev_groups = filter_weak_device_hints(fb_dev_labels, fb_dev_groups)
                fb_dev_labels, fb_dev_groups = reconcile_emulator_vs_real(fb_dev_labels, fb_dev_groups)
                if fb_dev_labels or fb_dev_groups:
                    dev_labels  = unique_preserve(dev_labels  + fb_dev_labels)
                    dev_groups  = unique_preserve(dev_groups  + fb_dev_groups)

            fb_trig_labels, fb_trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, fallback.lower())
            if any_match(GHA_GRADLE_INPUTS, fallback):
                fb_trig_labels.append("gha gradle arguments")
                fb_trig_groups.append("Gradle")
            if fb_trig_labels:
                trig_labels = unique_preserve(trig_labels + fb_trig_labels)
                trig_groups = unique_preserve(trig_groups + fb_trig_groups)

        has_device_setup = bool(dev_labels)
        has_test_trigger = bool(trig_labels)
        gradle_present   = bool(re.search(GRADLE_ANYWHERE, content))

        # --- Flutter device inference per job ---
        if "Flutter" in trig_groups:
            found = []
            # 1) Prefer explicit -d <device>
            for m in FLUTTER_CMD_LINE_RE.finditer(content):
                line = m.group(0)
                d = FLUTTER_DEVICE_FLAG_RE.search(line)
                if d:
                    plat = d.group("dev")
                    plat = plat.strip('"\'')
                    t = plat.lower()
                    if (t == "android" or t.startswith("emulator-") or
                        "sdk gphone" in t or "android sdk built for" in t or "pixel " in t):
                        found.append("android")
                    elif t in {"linux","macos","windows"}:
                        found.append(t)
                    elif t in {"ios","iphone","ipad","iphone simulator"}:
                        found.append("ios")
                    elif t in {"web","web-server","chrome","edge","firefox","safari"}:
                        found.append("web")
            # 2) iOS simulator hints
            if not found and any(p.search(job_raw) for p in IOS_SIM_HINTS):
                found.append("ios")
            # 3) Infer Android if strong emulator/GMD signals in the same job
            if not found and (set(dev_labels) & EMULATOR_STRONG_LABELS or "Emulator" in dev_groups or "GMD" in dev_groups):
                found.append("android")
            # 4) Linux headless hints
            if not found and LINUX_HEADLESS_HINTS_RE.search(content):
                found.append("linux")

            for plat in found:
                if plat and plat not in file_flutter_devices:
                    file_flutter_devices.append(plat)

        # --- Per-job decisive evidence flags (compute BEFORE aggregation uses them)
        has_emulator_group = any(g in {"Emulator", "GMD", "Third_Party_Lab"} for g in dev_groups)
        has_real_device_strong = any(lbl in REAL_DEVICE_STRONG_LABELS for lbl in dev_labels)

        # final job-level instru signal
        instru_t_ci_signal_job = bool(
            has_test_trigger or (has_device_setup and (has_emulator_group or has_real_device_strong))
        )
        file_instru_signal_any = file_instru_signal_any or instru_t_ci_signal_job

        # aggregate per-file
        file_device_labels  = unique_preserve(file_device_labels  + dev_labels)
        file_device_groups  = unique_preserve(file_device_groups  + dev_groups)
        file_trigger_labels = unique_preserve(file_trigger_labels + trig_labels)
        file_trigger_groups = unique_preserve(file_trigger_groups + trig_groups)
        file_has_test_trigger = file_has_test_trigger or has_test_trigger

        # AFTER (decisive device evidence: emulator/GMD/3P or strong real-device)
        if has_device_setup and (has_emulator_group or has_real_device_strong):
            file_has_device_with_group = True

        file_gradle_present_any = file_gradle_present_any or gradle_present

    # --- Confidence & reasons: include BOTH trigger & device when present ---
    reasons = []
    if file_has_test_trigger:
        reasons.append("test_trigger: " + ", ".join(file_trigger_labels))
    if file_has_device_with_group:
        msg = "device_setup: " + ", ".join(file_device_labels)
        if file_gradle_present_any:
            msg += "; gradle present"
        reasons.append(msg)
    reason = " | ".join(reasons)

    # simple confidence policy
    confidence = ""
    if file_has_test_trigger and file_has_device_with_group:
        confidence = "high"
    elif file_has_test_trigger or file_has_device_with_group:
        confidence = "medium"

    # AndroidTest presence boost
    if file_instru_signal_any and full_name in ANDROIDTEST_PRESENT:
        if confidence == "":
            confidence = "medium"
        elif confidence == "medium":
            confidence = "high"
        reason = (reason + " + boosted (AndroidTest present)").strip()

    # --- Derived style fields ---
    def map_execution_env(groups: List[str]) -> str:
        s = set(groups)
        if "GMD" in s: return "GMD"
        if "Emulator" in s: return "Emulator"
        if "Third_Party_Lab" in s: return "Third Party"
        if "Real_Device" in s: return "Real Device"
        return "Unknown"

    def map_test_invocation(groups: List[str]) -> str:
        s = set(groups)
        if "Third_Party_Lab" in s: return "3P CLIs"
        if "ADB" in s: return "ADB"
        if "Gradle" in s: return "Gradle"
        return "Unknown"

    rows.append({
        "filename": filename,
        "full_name": full_name,
        "ci_platform": ci_platform,
        "instru_t_ci_signal": bool(file_instru_signal_any),
        "confidence": confidence if file_instru_signal_any else "",
        "confidence_reason": reason if file_instru_signal_any else "",
        "execution_environment": map_execution_env(file_device_groups),
        "test_invocation": map_test_invocation(file_trigger_groups),
        "flutter_integ_t_signal": ("Flutter" in file_trigger_groups),
        "flutter_integ_t_d": ",".join(file_flutter_devices),
    })

# Save
out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV} (rows={len(out_df)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_YML_Files_V4.0.csv (rows=12667)


Instru Testing Signals from Build / Config files

In [ ]:
#Instru Test Signal Build / Config

import os
import re
import pandas as pd

# === CONFIG ===
ROOT = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_Instru_T_Signal_Config.csv"

# --- Helpers ---
def is_build_gradle_file(fname: str) -> bool:
    """
    Accept ONLY Gradle build files, including numbered variants saved with '++':
      - build.gradle / build.gradle.kts
      - build__<n>.gradle / build__<n>.gradle.kts
      - owner.repo__Type++build__<n>.gradle[.kts]
    """
    base = os.path.basename(fname)
    if "++" in base:
        tail = base.split("++", 1)[1]
        tail = re.sub(r"__\d+(?=\.gradle(?:\.kts)?$)", "", tail, flags=re.IGNORECASE).lower()
        return tail in ("build.gradle", "build.gradle.kts")
    return bool(re.match(r"(?i)^build(?:__\d+)?\.gradle(?:\.kts)?$", base))

def extract_full_name(fname: str, fpath: str) -> str:
    """Extract owner.repo from 'owner.repo__Type++...' filenames; fallback to parent dir."""
    base = os.path.basename(fname)
    if "__" in base:
        return base.split("__", 1)[0].lower()
    return os.path.basename(os.path.dirname(fpath)).lower()

def strip_comments_gradle(text: str) -> str:
    """Remove /* ... */ and // ... comments; keep http(s)://."""
    if not text:
        return ""
    s = re.sub(r"/\*.*?\*/", "", text, flags=re.DOTALL)
    s = re.sub(r"(?<!:)//.*?$", "", s, flags=re.MULTILINE)
    return s

def has_instru_signal_config(text: str) -> bool:
    """
    Instrumentation-test *config* signals in build files (not CI triggers).
    True if any strong signal is present:
      - androidTest dependencies
      - testInstrumentationRunner / args
      - GMD blocks (managedDevices / managedVirtualDevice / deviceGroups + apiLevel)
      - test-only module plugin: com.android.test
      - explicit androidComponents gating/enabling of androidTest
    """
    if not text:
        return False
    t = text.lower()

    # Strong: androidTest deps
    if ("androidtestimplementation" in t
        or "androidtestapi" in t
        or "androidtestcompileonly" in t
        or "androidtestruntimeonly" in t
        or "androidtestcompile" in t):
        return True

    # Runner / runner args
    if "testinstrumentationrunner" in t or "testinstrumentationrunnerarguments" in t:
        return True

    # GMD / managed devices (prefer pair: managed* + apiLevel)
    has_managed = ("manageddevices" in t) or ("managedvirtualdevice" in t) or ("devicegroups" in t)
    if has_managed and "apilevel" in t:
        return True

    # Test-only module plugin
    if 'id("com.android.test")' in t or "id 'com.android.test'" in t or 'apply plugin: "com.android.test"' in t:
        return True

    # Android Components gating (indicates androidTest consideration)
    if "enableandroidtest" in t:
        return True

    return False

# --- Scan & analyze ---
rows = []
for dirpath, _, files in os.walk(ROOT):
    for fname in files:
        if not is_build_gradle_file(fname):
            continue

        fpath = os.path.join(dirpath, fname)
        try:
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                raw = f.read()
        except Exception:
            raw = ""

        content = strip_comments_gradle(raw)
        signal = has_instru_signal_config(content)

        rows.append({
            "filename": os.path.basename(fpath),
            "full_name": extract_full_name(fname, fpath),
            "instru_t_signal_config": bool(signal),
        })

# --- Save ---
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(rows)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_Instru_T_Signal_Config.csv (rows=21280)
